<a href="https://colab.research.google.com/github/ksw9179/AI_and_Data-/blob/main/Bio_LLM(Protein_Language_Model)_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🛠️ 1단계: Bio-LLM 구동을 위한 필수 라이브러리 설치
Hugging Face의 transformers와 딥러닝 엔진인 torch가 필요

In [ ]:
# [1] Hugging Face Transformers 및 PyTorch 설치
!pip install transformers torch seaborn matplotlib -q

print("✅ Bio-LLM 구동을 위한 환경 구축이 완료되었습니다.")

### 🧬 2단계: 타겟 단백질 FASTA 서열 로드 및 Bio-LLM (ESM-2) 가동

단백질 서열(알파벳)을 AI가 이해할 수 있는 고차원 숫자 벡터(Embedding)로 변환하는 핵심 코드

In [ ]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, EsmModel
from sklearn.metrics.pairwise import cosine_similarity

print("🤖 [Bio-LLM] 메타(Meta)의 ESM-2 단백질 언어 모델 로딩 중...")

# 1. Hugging Face에서 단백질 특화 LLM인 ESM-2 모델과 토크나이저 불러오기
# (8M 파라미터를 가진 가벼운 모델을 사용하여 코랩 CPU에서도 빠르게 동작하도록 설정)
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name)

# 2. 타겟 단백질(HER2)과 비교군 단백질들의 아미노산 서열(FASTA 형태의 일부)
# 실제 NCBI 데이터베이스의 아미노산 서열 단편입니다.
protein_sequences = {
    "Target: HER2 (Cancer Growth)": "MELAALCRWGLLLALLPPGAASTQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQ",
    "Compare 1: EGFR (Same Family)": "MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNM",
    "Compare 2: PD-L1 (Immune Marker)": "MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGV",
    "Compare 3: Insulin (Metabolism)": "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"
}

# 3. 단백질 서열을 임베딩(고차원 벡터)으로 변환하는 함수 구축
def get_protein_embedding(sequence):
    # 아미노산 서열을 토큰화 (문자를 숫자로)
    inputs = tokenizer(sequence, return_tensors="pt", padding=True, truncation=True)

    # 모델 통과 (기울기 계산 비활성화 no_grad()로 속도 향상)
    with torch.no_grad():
        outputs = model(**inputs)

    # 단백질 전체 서열을 대표하는 하나의 벡터 추출 (Mean Pooling)
    # 마지막 히든 스테이트의 토큰 벡터들의 평균을 구함
    sequence_embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()     # 인공지능이 계산을 마치면 outputs.last_hidden_state 결과가 나옴, 아미노산 글자 하나하나마다 인공지능이 생각한 의미 점수
    return sequence_embedding

# 4. 각 단백질의 임베딩 벡터 추출
embeddings = {}
for name, seq in protein_sequences.items():
    embeddings[name] = get_protein_embedding(seq)

print("✅ 단백질 서열 토큰화 및 고차원 임베딩 추출 완료!")

In [ ]:
print(embeddings)

### 📊 3단계: 임베딩 벡터 간의 기능적 유사도(Cosine Similarity) 비교
LLM이 추출한 벡터를 바탕으로, HER2가 어떤 단백질과 생물학적으로 가장 닮아있는지 히트맵으로 시각화.

In [ ]:
print("🔍 [분석] 추출된 임베딩을 바탕으로 단백질 간 진화적/기능적 유사도 비교")

# 1. 유사도 행렬 계산을 위해 리스트 형태로 변환
names = list(embeddings.keys())
vector_list = list(embeddings.values())

# 2. 코사인 유사도(Cosine Similarity) 계산
similarity_matrix = cosine_similarity(vector_list)

# 3. 시각화 (Heatmap)
plt.figure(figsize=(8, 6))
sns.heatmap(similarity_matrix, xticklabels=names, yticklabels=names,
            annot=True, cmap="YlGnBu", fmt=".3f", linewidths=.5)
plt.title("Protein Semantic Similarity by Bio-LLM (ESM-2)", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.show()

# 4. 분석 결과 텍스트 출력
target_name = names[0]
most_similar_idx = np.argsort(similarity_matrix[0])[::-1][1] # 자기 자신(1.0) 제외 1등
print(f"🎯 [AI 결론] Bio-LLM 분석 결과, '{target_name}'는 ")
print(f"'{names[most_similar_idx]}'와 가장 높은 기능적/구조적 유사도를 보입니다.")

#💡 Bio-AI 핵심 개념 (Bottom-up 개념 정리)
위 코드가 어떤 원리로 작동하는지 레포트나 세션 발표에 활용할 수 있도록 개념을 정리.

## Hugging Face & ESM-2 모델 로딩

자연어 처리에서는 단어나 형태소를 쪼개지만,

Bio-LLM에서는 M, E, L, A 등 개별 아미노산을 하나의 토큰으로 인식.

tokenizer가 이 알파벳들을 모델이 계산할 수 있는 고유한 숫자로 바꿔주는 역할.


### *ESM-2 (Evolutionary Scale Modeling)*
: 메타(Meta) 연구진이 수억 개의 단백질 서열을 학습시켜 만든 모델.

인간의 언어를 학습한 ChatGPT처럼, 아미노산의 배열 규칙(생명의 문법)을 학습해서 단백질의 3차원 구조나 기능적 속성을 예측.

### *토큰화 (Tokenization)*

자연어 처리에서는 단어나 형태소를 쪼개지만,

Bio-LLM에서는 M, E, L, A 등 개별 아미노산을 하나의 토큰으로 인식.

tokenizer가 이 알파벳들을 모델이 계산할 수 있는 고유한 숫자로 바꿔주는 역할.

### *임베딩 (Embedding)*

단백질 서열이 모델의 Self-Attention 층을 통과하면,

단순한 문자가 아니라 주변 아미노산들과의 관계가 반영된 '고차원 벡터(숫자 배열)'로 변환.

위 코드에서는 수백 개의 아미노산 토큰을 평균 내어(Mean Pooling) 단백질 하나를 대표하는 지문(벡터)을 만들었어.

### *유사도 비교 (Cosine Similarity)*

각 단백질의 벡터 방향이 얼마나 일치하는지 각도(Cosine)를 측정.

결과 히트맵을 보면, 암 증식 타겟인 HER2는 같은 세포 수용체인 EGFR과는 매우 높은 유사도(언어적 패턴이 비슷함)를 보이지만,

인슐린 같은 대사 단백질과는 전혀 다른 벡터 공간에 위치한다는 것을 AI가 서열만 보고 스스로 판단.

# Self-Attention
아주 핵심적인 딥러닝 개념

 챗GPT나 메타의 ESM-2 같은 모든 트랜스포머(Transformer) 기반 AI 모델들이 천재적인 능력을 발휘하게 만드는 가장 핵심적인 '생각의 원리'.

## 💡 1. 셀프 어텐션(Self-Attention)의 진짜 뜻: "서로(Self) 주목하기(Attention)"
'Attention'은 영어로 '집중' 또는 '주목'이라는 뜻

 셀프 어텐션은 인공지능이 문장(또는 단백질 서열)을 읽을 때, "어떤 글자와 어떤 글자가 서로 깊은 연관이 있는지 스스로 집중해서 쳐다보는 능력"이야.

### 인간의 언어로 먼저 비유

"어제 길을 가다가 *귀여운 강아지*를 봤는데, *그 녀석* 이 나를 보고 꼬리를 흔들었어."

우리는 이 문장을 읽을 때 *그 녀석* 이라는 단어를 보면, 시키지 않아도 뇌 속에서 자동으로 앞에 나왔던 강아지와 연결

인공지능도 마찬가지. 단순히 글자를 기계적으로 읽는 게 아니라,

문장 안의 모든 단어를 서로서로 대조해 보면서 "아! 여기서 '그 녀석'은 '강아지'를 강하게 주목(Attention)해야 문맥이 통하는구나!" 하고 스스로 알아챔.

이 연결 연산을 수행하는 방이 Self-Attention 층.

## 🧬 2. 단백질 세계에서 Self-Attention이 왜 혁명적일까?
이 원리를 단백질 아미노산 서열에 그대로 가져오기.

단백질은 알파벳이 일렬로 길게 늘어선 문자열(예: M-E-L-A...)처럼 보이지만, 실제 세포 안에서는 가만히 펴져 있지 않고 지리멸렬하게 꼬이고 접혀서 3차원 입체 로봇 모양

이때, 서열 상으로는 1번째에 있는 아미노산(M)과 저 뒤에 멀리 떨어진 80번째 아미노산(Q)이 입체적으로 접히면서 실제로는 서로 딱 달라붙어 중요한 기능을 할 수 있음.

### Self-Attention 층이 없다면?

AI는 1번째 글자와 80번째 글자가 멀리 떨어져 있으니까 아무 상관이 없다고 생각할 거야. 마치 문맥을 못 읽는 컴퓨터처럼 말이지.

### Self-Attention 층이 있다면?

AI는 1번째 글자부터 마지막 글자까지 전부 자기들끼리 짝을 지어보면서 "오라? 1번 아미노산이랑 80번 아미노산이 생물학적 문법상 아주 끈끈하게 연결되어 있네! 얘네 둘이 집중해서 봐야겠다!" 하고 관계 점수를 높게 매겨버림.

## 🎯 3. 요약: 단순한 알파벳이 '고차원 벡터'가 되는 과정
처음 입력된 아미노산 M은 그냥 'M'이라는 *단순한 글자 번호표* 일 뿐.

하지만 Self-Attention 층을 통과하고 나면, 이 M은 그냥 M이 아니라

"뒤에 나오는 E와는 이런 관계고, 저 멀리 접히는 Q와는 강력하게 결합하는 성질을 가진 M"이라는 엄청나게 깊은 **생물학적 문맥(관계성)** 가짐.

이렇게 주변 아미노산들과의 입체적인 관계 점수들이 빽빽하게 채워진 숫자의 덩어리가 바로 '고차원 벡터(임베딩)'.